<a href="https://colab.research.google.com/github/aisha13dikko-sudo/using-synthetic-data-for-thermal-comfort-classification/blob/main/wk14_llm_subjectwise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# wk14: Static LLM classification on the subject-wise split

## Why this notebook exists

`LLM_Thermal_Comfort_Classification.ipynb` evaluated GPT-4o-mini on the
AutoTherm **built-in** test split (participants 5 and 12, 194,829 rows, Cold = 0.60%).
All tabular experiments in this project use a **subject-wise** split (participants
14, 16 and 20 held out, 290,019 rows, Cold = 7.78%).

Macro F1 computed on one is not comparable to macro F1 computed on the other, so the
reported result that few-shot 7-class GPT-4o-mini (0.3142) exceeded the Random Forest
baseline (0.2858) could not be defended.

This notebook re-runs all four static conditions on the subject-wise split so the
comparison is valid.

## Critical design point

Few-shot examples are drawn **only from the 13 training participants**. The original
notebook drew from `dataset["train"]`, which contains participants 14, 16 and 20. Under
the subject-wise split those are the test participants, so drawing examples from them
would place test-participant labels into the prompt. That is leakage and it is asserted
against below.

## Conditions

| # | Granularity | Prompting | Test sample | Few-shot examples |
|---|---|---|---|---|
| 1 | 3-class | zero-shot | 300 (100/class) | none |
| 2 | 3-class | few-shot   | 300 (100/class) | 3 per class |
| 3 | 7-class | zero-shot | 350 (50/class)  | none |
| 4 | 7-class | few-shot   | 350 (50/class)  | 2 per class |

Sample sizes match the original notebook so the two are comparable in design.

**Cost:** ~1,300 calls, roughly $0.10. **Time:** 25-35 minutes.


## Cell 1 — install and API key

Your previous OpenAI key was revoked after being found hardcoded in a notebook. Set the
new one as a Colab secret: click the **key icon** in the left sidebar, add a secret named
`OPENAI_API_KEY`, paste the value, enable notebook access. Never type a key into a cell.

In [1]:
!pip install -q openai datasets 2>&1 | tail -2

from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
print("Client ready.")

Client ready.


## Cell 2 — constants and the result logger

`log_result` is the single point of truth. Nothing in this notebook is ever typed into a
summary table by hand. That habit is what produced the CTGAN discrepancy in wk3.

In [2]:
import os, re, json, time, platform, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings("ignore")
os.makedirs("results", exist_ok=True)

RANDOM_STATE = 42
MODEL = "gpt-4o-mini"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
TEST_PARTICIPANTS = ["participant_14", "participant_16", "participant_20"]

# The seven features described to the model, matching the original notebook.
LLM_FEATURES = ["Ambient_Temperature", "Wrist_Skin_Temperature", "Radiation-Temp",
                "Ambient_Humidity", "GSR", "Heart_Rate", "Bodytemp"]

LABEL_NAMES_7 = {-3: "Cold", -2: "Cool", -1: "Slightly Cool", 0: "Neutral",
                 1: "Slightly Warm", 2: "Warm", 3: "Hot"}
LABEL_NAMES_3 = {-1: "Cold", 0: "Neutral", 1: "Warm"}

MANIFEST = {"run_id": RUN_ID,
            "started": datetime.now().isoformat(timespec="seconds"),
            "python": platform.python_version(), "pandas": pd.__version__,
            "sklearn": sklearn.__version__, "model": MODEL,
            "temperature": 0, "random_state": RANDOM_STATE,
            "purpose": "re-run static LLM classification on the subject-wise split"}

RESULTS = []
PREDICTIONS = {}

def log_result(condition, granularity, y_true, y_pred, n_calls, elapsed_s, notes=""):
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cold_label = -3 if granularity == 7 else -1
    cold = f1_score(y_true, y_pred, labels=[cold_label],
                    average="macro", zero_division=0)
    row = {"condition": condition, "granularity": granularity,
           "macro_f1": round(float(macro), 4), "cold_f1": round(float(cold), 4),
           "n_calls": n_calls, "elapsed_s": round(elapsed_s, 1),
           "ms_per_call": round(1000 * elapsed_s / n_calls, 1),
           "split": "subject-wise", "run_id": RUN_ID, "notes": notes}
    RESULTS.append(row)
    print(f"\n  {condition:<28} {granularity}-class  "
          f"macro_f1={macro:.4f}  cold_f1={cold:.4f}  ({n_calls} calls, {elapsed_s:.0f}s)")
    return row

print("Run ID:", RUN_ID)

Run ID: 20260811_235539


## Cell 3 — subject-wise split

Identical to wk9, wk10, wk12 and wk13. The assertions stop the notebook if anything
upstream has changed, because nothing below would then be comparable.

In [3]:
from datasets import load_dataset

dataset = load_dataset("kopetri/AutoTherm", "indoor")
df = dataset["train"].to_pandas()

df["participant_id"] = df["file_name"].apply(
    lambda f: re.search(r"participant_\d+", f).group())
df["Label_3class"] = df["Label"].apply(
    lambda x: -1 if x <= -2 else (0 if x <= 1 else 1))

train_df = df[~df["participant_id"].isin(TEST_PARTICIPANTS)].copy()
test_df  = df[ df["participant_id"].isin(TEST_PARTICIPANTS)].copy()

assert len(train_df) == 1_276_709, "train rows differ from the other notebooks"
assert len(test_df)  ==   290_019, "test rows differ from the other notebooks"

# LEAKAGE GATE. Few-shot examples come from train_df only.
assert not set(train_df["participant_id"]) & set(TEST_PARTICIPANTS), \
    "test participants present in the few-shot example pool"

print(f"Train: {len(train_df):,} rows, {train_df['participant_id'].nunique()} participants")
print(f"Test:  {len(test_df):,} rows,  {sorted(test_df['participant_id'].unique())}")
print(f"\nCold (-3) in test: {(test_df['Label']==-3).sum():,} "
      f"({100*(test_df['Label']==-3).mean():.2f}%)")
print("\nTest label distribution (7-class):")
print(test_df["Label"].value_counts().sort_index().to_string())

README.md:   0%|          | 0.00/8.57k [00:00<?, ?B/s]

indoor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 29.8MB            

indoor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

indoor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B / 30.0MB            

indoor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

indoor/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.41MB            

indoor/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1566728 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/194829 [00:00<?, ? examples/s]

Train: 1,276,709 rows, 13 participants
Test:  290,019 rows,  ['participant_14', 'participant_16', 'participant_20']

Cold (-3) in test: 22,556 (7.78%)

Test label distribution (7-class):
Label
-3    22556
-2    19285
-1    85260
 0    39044
 1    16824
 2    51786
 3    55264


## Cell 4 — stratified test samples

Sample sizes match the original notebook: 100 per class at 3-class, 50 per class at
7-class. `random_state=42` so the same rows are drawn on any re-run.

In [4]:
def stratified_sample(frame, label_col, per_class, seed=RANDOM_STATE):
    return (frame.groupby(label_col, group_keys=False)
            .apply(lambda g: g.sample(n=min(len(g), per_class), random_state=seed))
            .reset_index(drop=True))

sample_3 = stratified_sample(test_df, "Label_3class", 100)
sample_7 = stratified_sample(test_df, "Label", 50)

print(f"3-class test sample: {len(sample_3)} rows")
print(sample_3["Label_3class"].value_counts().sort_index().to_string())
print(f"\n7-class test sample: {len(sample_7)} rows")
print(sample_7["Label"].value_counts().sort_index().to_string())

# Few-shot example pools, drawn from TRAINING participants only.
fewshot_3 = stratified_sample(train_df, "Label_3class", 3, seed=RANDOM_STATE)
fewshot_7 = stratified_sample(train_df, "Label", 2, seed=RANDOM_STATE)

assert set(fewshot_3["participant_id"]).isdisjoint(TEST_PARTICIPANTS)
assert set(fewshot_7["participant_id"]).isdisjoint(TEST_PARTICIPANTS)
print(f"\nFew-shot pools: {len(fewshot_3)} rows (3-class), {len(fewshot_7)} rows (7-class)")
print("Leakage gate passed: no test participants in either pool.")

3-class test sample: 300 rows
Label_3class
-1    100
 0    100
 1    100

7-class test sample: 350 rows
Label
-3    50
-2    50
-1    50
 0    50
 1    50
 2    50
 3    50

Few-shot pools: 9 rows (3-class), 14 rows (7-class)
Leakage gate passed: no test participants in either pool.


## Cell 5 — prompt construction and the API call

`temperature=0` for determinism. Any response that cannot be parsed to a valid label is
recorded as a parse failure and counted, rather than silently dropped, because silently
dropping failures would inflate the score.

In [5]:
def describe_row(row):
    return ", ".join(f"{f.replace('_',' ')}={row[f]:.2f}"
                     for f in LLM_FEATURES if pd.notna(row[f]))

def build_prompt(row, granularity, examples=None):
    names = LABEL_NAMES_7 if granularity == 7 else LABEL_NAMES_3
    label_col = "Label" if granularity == 7 else "Label_3class"
    options = ", ".join(f"{k} ({v})" for k, v in sorted(names.items()))

    p = ("You are classifying a person's thermal comfort from wearable sensor "
         f"readings.\n\nRespond with exactly one integer from: {options}.\n"
         "Respond with the integer only. No explanation.\n\n")

    if examples is not None and len(examples):
        p += "Examples:\n"
        for _, ex in examples.iterrows():
            p += f"{describe_row(ex)} -> {ex[label_col]}\n"
        p += "\n"

    p += f"Reading: {describe_row(row)}\nLabel:"
    return p


def classify(rows, granularity, examples, condition):
    label_col = "Label" if granularity == 7 else "Label_3class"
    valid = set(LABEL_NAMES_7 if granularity == 7 else LABEL_NAMES_3)
    y_true, y_pred, failures = [], [], 0
    t0 = time.time()

    for i, (_, row) in enumerate(rows.iterrows(), 1):
        try:
            r = client.chat.completions.create(
                model=MODEL, temperature=0, max_tokens=5,
                messages=[{"role": "user",
                           "content": build_prompt(row, granularity, examples)}])
            m = re.search(r"-?\d+", r.choices[0].message.content)
            pred = int(m.group()) if m else None
            if pred not in valid:
                pred, failures = None, failures + 1
        except Exception as e:
            print(f"  [call {i}] {e}")
            pred, failures = None, failures + 1

        # An unparseable response is scored as the majority class rather than
        # discarded. Discarding it would remove a case the model got wrong.
        if pred is None:
            pred = 0
        y_true.append(row[label_col]); y_pred.append(pred)

        if i % 50 == 0:
            print(f"    {condition}: {i}/{len(rows)}", flush=True)

    elapsed = time.time() - t0
    PREDICTIONS[condition] = {"y_true": y_true, "y_pred": y_pred}
    log_result(condition, granularity, y_true, y_pred, len(rows), elapsed,
               notes=f"{failures} parse failures")
    print(classification_report(y_true, y_pred, zero_division=0))
    return y_true, y_pred

print("Ready.")

Ready.


## Cell 6 — run all four conditions

25 to 35 minutes. Progress prints every 50 calls.

In [6]:
print("=== 1/4  3-class zero-shot ===")
classify(sample_3, 3, None, "3-class zero-shot")

print("\n=== 2/4  3-class few-shot (3 examples per class) ===")
classify(sample_3, 3, fewshot_3, "3-class few-shot")

print("\n=== 3/4  7-class zero-shot ===")
classify(sample_7, 7, None, "7-class zero-shot")

print("\n=== 4/4  7-class few-shot (2 examples per class) ===")
classify(sample_7, 7, fewshot_7, "7-class few-shot")

print("\nAll four conditions complete.")

=== 1/4  3-class zero-shot ===
    3-class zero-shot: 50/300
    3-class zero-shot: 100/300
    3-class zero-shot: 150/300
    3-class zero-shot: 200/300
    3-class zero-shot: 250/300
    3-class zero-shot: 300/300

  3-class zero-shot            3-class  macro_f1=0.4620  cold_f1=0.0000  (300 calls, 138s)
              precision    recall  f1-score   support

          -1       0.00      0.00      0.00       100
           0       0.41      0.72      0.52       100
           1       0.78      0.97      0.86       100

    accuracy                           0.56       300
   macro avg       0.40      0.56      0.46       300
weighted avg       0.40      0.56      0.46       300


=== 2/4  3-class few-shot (3 examples per class) ===
    3-class few-shot: 50/300
    3-class few-shot: 100/300
    3-class few-shot: 150/300
    3-class few-shot: 200/300
    3-class few-shot: 250/300
    3-class few-shot: 300/300

  3-class few-shot             3-class  macro_f1=0.4258  cold_f1=0.4242  (300

In [7]:
# Baseline evaluated on the identical stratified samples the LLM sees.
# Without this, LLM macro F1 (balanced 350-row sample) is being compared to
# RF macro F1 (full 290,019-row test set at natural class prevalence), and
# macro F1 is not invariant to test-set class distribution.

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

DROP_COLS = ["file_name", "Timestamp", "participant_id",
             "Air-Velocity", "Metabolic-Rate",
             "Nose", "Neck", "RShoulder", "RElbow",
             "LShoulder", "LElbow", "REye", "LEye", "REar", "LEar",
             "Emotion-Self", "Emotion-ML", "Label", "Label_3class"]

def prep(frame):
    X = frame.drop(columns=[c for c in DROP_COLS if c in frame.columns],
                   errors="ignore").copy()
    if "Gender" in X.columns:
        X["Gender"] = LabelEncoder().fit_transform(X["Gender"].astype(str))
    return X.select_dtypes(include=[np.number])

X_train = prep(train_df)
assert "Label" not in X_train.columns and "Label_3class" not in X_train.columns
print(f"{X_train.shape[1]} features")

for gran, tcol, sample in [(7, "Label", sample_7), (3, "Label_3class", sample_3)]:
    clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                 n_jobs=-1).fit(X_train, train_df[tcol])
    Xs = prep(sample).reindex(columns=X_train.columns)
    log_result(f"RF baseline (LLM sample)", gran, sample[tcol], clf.predict(Xs),
               n_calls=len(sample), elapsed_s=0.001,
               notes="same stratified rows as the LLM conditions")
    print(classification_report(sample[tcol], clf.predict(Xs),
                                digits=4, zero_division=0))

18 features

  RF baseline (LLM sample)     7-class  macro_f1=0.2708  cold_f1=0.0000  (350 calls, 0s)
              precision    recall  f1-score   support

          -3     0.0000    0.0000    0.0000        50
          -2     0.1270    0.1600    0.1416        50
          -1     0.3645    0.7800    0.4968        50
           0     0.1833    0.2200    0.2000        50
           1     0.3256    0.2800    0.3011        50
           2     0.2807    0.3200    0.2991        50
           3     0.8000    0.3200    0.4571        50

    accuracy                         0.2971       350
   macro avg     0.2973    0.2971    0.2708       350
weighted avg     0.2973    0.2971    0.2708       350


  RF baseline (LLM sample)     3-class  macro_f1=0.7050  cold_f1=0.7261  (300 calls, 0s)
              precision    recall  f1-score   support

          -1     1.0000    0.5700    0.7261       100
           0     0.5275    0.9600    0.6809       100
           1     0.9344    0.5700    0.7081     

## Cell 7 — compare against the baseline, now on the same split

In [8]:
res = pd.DataFrame(RESULTS)

# Pull the sample-matched baselines out of RESULTS rather than hardcoding them.
# This is the same discipline as log_result: a summary that reads from the
# computation cannot contradict it.
base = {}
for _, r in res[res["condition"].str.startswith("RF baseline")].iterrows():
    base[r["granularity"]] = r["macro_f1"]

if not base:
    raise RuntimeError("No RF baseline rows found. Run the baseline cell first.")

print(res.to_string(index=False))

print("\n" + "=" * 70)
print("LLM vs RANDOM FOREST — same participants, same rows, same class balance")
print("=" * 70)
for _, r in res[~res["condition"].str.startswith("RF baseline")].iterrows():
    b = base[r["granularity"]]
    d = r["macro_f1"] - b
    print(f"  {r['condition']:<24} {r['macro_f1']:.4f} vs {b:.4f}  "
          f"{d:+.4f}  {'BEATS BASELINE' if d > 0 else 'below'}")

print("\nCold F1 (7-class label -3; 3-class merged {-3,-2}):")
for _, r in res.iterrows():
    print(f"  {r['condition']:<24} {r['granularity']}-class  {r['cold_f1']:.4f}")

print("\n" + "-" * 70)
print("For reference only, NOT a valid comparison:")
print("  RF on the full 290,019-row test set: 7-class 0.2858, 3-class 0.7163")
print("  Original LLM figures on the built-in split (participants 5 and 12,")
print("  Cold prevalence 0.60%): 3c zero-shot 0.4447, 3c few-shot 0.3154,")
print("  7c zero-shot 0.1552, 7c few-shot 0.3142")
print("-" * 70)

               condition  granularity  macro_f1  cold_f1  n_calls  elapsed_s  ms_per_call        split          run_id                                      notes
       3-class zero-shot            3    0.4620   0.0000      300      137.7        459.2 subject-wise 20260811_235539                           0 parse failures
        3-class few-shot            3    0.4258   0.4242      300      141.3        471.1 subject-wise 20260811_235539                           0 parse failures
       7-class zero-shot            7    0.2475   0.0000      350      158.5        452.9 subject-wise 20260811_235539                           0 parse failures
        7-class few-shot            7    0.2216   0.0000      350      176.7        504.8 subject-wise 20260811_235539                           0 parse failures
RF baseline (LLM sample)            7    0.2708   0.0000      350        0.0          0.0 subject-wise 20260811_235539 same stratified rows as the LLM conditions
RF baseline (LLM sample)    

## Cell 8 — save and download immediately


In [9]:
MANIFEST["finished"] = datetime.now().isoformat(timespec="seconds")
MANIFEST["n_conditions"] = len(RESULTS)
MANIFEST["test_participants"] = TEST_PARTICIPANTS
MANIFEST["n_test_rows"] = int(len(test_df))
MANIFEST["cold_prevalence_test_pct"] = round(100 * (test_df["Label"] == -3).mean(), 2)
MANIFEST["sample_sizes"] = {"3-class": len(sample_3), "7-class": len(sample_7)}
MANIFEST["fewshot_examples_per_class"] = {"3-class": 3, "7-class": 2}
MANIFEST["fewshot_source"] = "13 training participants only; test participants asserted absent"

pd.DataFrame(RESULTS).to_csv("results/wk14_llm_subjectwise.csv", index=False)
with open("results/wk14_manifest.json", "w") as f:
    json.dump(MANIFEST, f, indent=2)
with open("results/wk14_predictions.json", "w") as f:
    json.dump({k: {kk: [int(x) for x in vv] for kk, vv in v.items()}
               for k, v in PREDICTIONS.items()}, f, indent=2)

from google.colab import files
for fn in ["wk14_llm_subjectwise.csv", "wk14_manifest.json", "wk14_predictions.json"]:
    files.download(f"results/{fn}")
print("Saved and downloaded.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved and downloaded.
